# **Klasifikasi Bentuk Wajah & Rekomendasi Gaya Rambut**
### **PBL Semester 5 - Pipeline Pelatihan Machine Learning (6 Kelas)**

Alur kerja (*workflow*) pelatihan:
1. **Ekstraksi Dataset**: Mengambil data wajah dari `faceShape_6classes.zip` (folder `train/`).
2. **Deteksi Landmark Wajah (MediaPipe FaceLandmarker)**: Mendeteksi 478 titik landmark wajah dengan Google MediaPipe Tasks API dan memotong wajah secara proporsional bujur sangkar (1:1).
3. **Ekstraksi Fitur HOG (Histogram of Oriented Gradients)**: Mengubah citra ke 128x128 Grayscale dan mengekstrak vektor fitur 8.100 dimensi.
4. **Pelatihan Model**: Melatih model Support Vector Machine (SVM) dan Random Forest Classifier pada 6 kelas bentuk wajah (**Diamond, Heart, Oblong, Oval, Round, Square**).

## **1. Install MediaPipe, Hubungkan Google Drive & Ekstrak Dataset**
Menginstall `mediapipe` dan mengunduh model resmi `face_landmarker.task` dari Google Storage.

In [ ]:
# Install MediaPipe & unduh model resmi FaceLandmarker
!pip install -q mediapipe
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

import os
import zipfile
from google.colab import drive

# Hubungkan ke Google Drive
drive.mount('/content/drive')

EXTRACT_DIR = '/content/dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# Ekstrak file zip terpadu (faceShape_6classes.zip)
ZIP_PATH = '/content/drive/MyDrive/faceShape/faceShape_6classes.zip'
print(f"Mengekstrak dataset: {ZIP_PATH}...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

TRAIN_DIR = os.path.join(EXTRACT_DIR, 'train')
print(f"Dataset berhasil diekstrak! Folder train: {TRAIN_DIR}")
print("Daftar kelas:", sorted(os.listdir(TRAIN_DIR)))

## **2. Deteksi Landmark Wajah (Google MediaPipe FaceLandmarker)**
Menggunakan MediaPipe Tasks API (`vision.FaceLandmarker`) untuk mendeteksi 478 titik landmark struktural wajah dan memotong wajah bujur sangkar 1:1.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Inisialisasi MediaPipe FaceLandmarker (478 landmark)
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    min_face_detection_confidence=0.5
)
face_landmarker = vision.FaceLandmarker.create_from_options(options)

def detect_and_crop_face(image_pil, padding=0.20):
    """
    Mendeteksi wajah menggunakan 478 landmark MediaPipe dan memotong bujur sangkar 1:1
    dengan margin padding 20% agar kontur dahi, tulang pipi, dan dagu tetap utuh.
    """
    img_rgb = np.array(image_pil.convert('RGB'))
    h, w, _ = img_rgb.shape
    
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    results = face_landmarker.detect(mp_image)
    if results.face_landmarks:
        landmarks = results.face_landmarks[0]
        x_coords = [lm.x * w for lm in landmarks]
        y_coords = [lm.y * h for lm in landmarks]
        
        xmin, xmax = min(x_coords), max(x_coords)
        ymin, ymax = min(y_coords), max(y_coords)
        
        bw = xmax - xmin
        bh = ymax - ymin
        
        cx = xmin + bw / 2.0
        cy = ymin + bh / 2.0
        sq_size = max(bw, bh) * (1.0 + padding * 2)
        
        x1 = max(0, int(cx - sq_size / 2.0))
        y1 = max(0, int(cy - sq_size / 2.0))
        x2 = min(w, int(cx + sq_size / 2.0))
        y2 = min(h, int(cy + sq_size / 2.0))
        
        return image_pil.crop((x1, y1, x2, y2))
    
    return image_pil

# Visualisasi contoh landmark MediaPipe
sample_class = 'diamond'
sample_dir = os.path.join(TRAIN_DIR, sample_class)
sample_file = os.listdir(sample_dir)[0]
sample_im = Image.open(os.path.join(sample_dir, sample_file))

demo_rgb = np.array(sample_im.convert('RGB'))
mp_demo = mp.Image(image_format=mp.ImageFormat.SRGB, data=demo_rgb)
res_demo = face_landmarker.detect(mp_demo)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title("Original + 478 Landmark MediaPipe")
demo_plot = demo_rgb.copy()
if res_demo.face_landmarks:
    for lm in res_demo.face_landmarks[0]:
        px, py = int(lm.x * demo_plot.shape[1]), int(lm.y * demo_plot.shape[0])
        cv2.circle(demo_plot, (px, py), 1, (0, 255, 0), -1)
plt.imshow(demo_plot)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Hasil MediaPipe Auto-Crop 1:1")
cropped_demo = detect_and_crop_face(sample_im)
plt.imshow(cropped_demo)
plt.axis('off')
plt.tight_layout()
plt.show()

## **3. Ekstraksi Fitur HOG (Histogram of Oriented Gradients)**
Gambar wajah hasil *crop* MediaPipe diubah ke Grayscale, di-*resize* ke 128x128 piksel, dinormalisasi 0..1, lalu diekstrak fitur HOG (8.100 dimensi).

In [ ]:
from skimage.feature import hog
from tqdm import tqdm

# 6 kelas bentuk wajah
CLASSES = ['diamond', 'heart', 'long', 'oval', 'round', 'square']

def extract_hog_features(image_path):
    # 1. Buka gambar
    raw_img = Image.open(image_path)
    
    # 2. Deteksi & crop wajah dengan MediaPipe FaceLandmarker
    face_crop = detect_and_crop_face(raw_img)
    
    # 3. Resize ke 128x128 Grayscale
    gray_img = face_crop.convert('L').resize((128, 128), Image.Resampling.LANCZOS)
    gray_arr = np.array(gray_img, dtype=np.uint8)

    # 4. Adaptive Contrast Enhancement (CLAHE) - menyeimbangkan kontras/backlight
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_arr = clahe.apply(gray_arr)
    img_array = enhanced_arr.astype(np.float32) / 255.0

    # 4. Ekstraksi fitur HOG (8.100 dimensi)
    features = hog(
        img_array,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        transform_sqrt=True,
        feature_vector=True
    )
    return features

# Batas sampel seimbang (1.000 gambar per kelas = total 6.000 gambar)
MAX_SAMPLES_PER_CLASS = 1000

X = []
y = []

print("Mengekstrak fitur HOG dengan MediaPipe FaceLandmarker...")
for cls_name in CLASSES:
    class_dir = os.path.join(TRAIN_DIR, cls_name)
    canonical_name = 'oblong' if cls_name == 'long' else cls_name
    
    all_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    selected_files = all_files[:MAX_SAMPLES_PER_CLASS]
    print(f"Memproses kelas '{canonical_name}' ({cls_name}): {len(selected_files)} gambar...")

    for fname in tqdm(selected_files):
        fpath = os.path.join(class_dir, fname)
        feat = extract_hog_features(fpath)
        if feat is not None:
            X.append(feat)
            y.append(canonical_name)

X = np.array(X)
y = np.array(y)
print(f"\nEkstraksi Selesai! Total sampel: {len(X)} | Dimensi HOG: {X.shape[1]}")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f" - {u.capitalize()}: {c} data")

## **4. Split Data (Train 80% & Test 20%)**
Pembagian data secara acak terstratifikasi (*Stratified Split*) agar proporsi 6 kelas tetap seimbang.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Jumlah Data Latih (Train) : {len(X_train)}")
print(f"Jumlah Data Uji (Test)    : {len(X_test)}")

## **5. Training & Evaluasi Model: SVM vs Random Forest**
Melatih model Support Vector Machine (Kernel RBF) dan Random Forest Classifier (100 Trees).

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# 1. Model Support Vector Machine (SVM)
print("Melatih model SVM (kernel RBF, probability=True)...\n")
svm_model = SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f"Akurasi Model SVM: {svm_acc * 100:.2f}%")

# 2. Model Random Forest
print("Melatih model Random Forest (100 trees)...\n")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Akurasi Model Random Forest: {rf_acc * 100:.2f}%")

# 3. Laporan Evaluasi SVM
print("\n=== CLASSIFICATION REPORT (SVM) ===")
print(classification_report(y_test, svm_preds))

# 4. Visualisasi Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, svm_preds, labels=svm_model.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=svm_model.classes_, yticklabels=svm_model.classes_)
plt.title(f"Confusion Matrix SVM 6 Kelas (Akurasi: {svm_acc * 100:.2f}%)")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.show()

## **6. Export & Download Model Artifact (`face_shape_svm.pkl`)**
Simpan model SVM terlatih sebagai `face_shape_svm.pkl` untuk backend web.

In [ ]:
import joblib
from google.colab import files

MODEL_FILE = 'face_shape_svm.pkl'
joblib.dump(svm_model, MODEL_FILE)
print(f"Model berhasil disimpan sebagai '{MODEL_FILE}'!")

# Download file model ke komputer
files.download(MODEL_FILE)